# AIMS LTMP Fish Occurrence — EDA

This notebook provides a lightweight exploratory analysis of `AIMSdataset/occurrence.txt` (Darwin Core format).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

path = Path('AIMSdataset/occurrence.txt')

df = pd.read_csv(path, sep='	', low_memory=False)
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')

df.head()

## Dataset summary
- **Rows:** 168,620
- **Columns:** 66
- **Date range:** 1992-03-28 to 2018-05-04
- **Unique scientific names:** 253
- **Unique families:** 11
- **Unique genera:** 54
- **Countries:** 1
- **States/Provinces:** 1
- **Latitude range:** -23.89088 to -14.52352
- **Longitude range:** 145.35140 to 152.66857

## Data quality (missingness)

Top 10 fields by missing percentage:

| item | % missing |
|---|---|
| references | 100.0 |
| bibliographicCitation | 100.0 |
| typeStatus | 100.0 |
| subgenus | 100.0 |
| materialSampleID | 100.0 |
| lifeStage | 100.0 |
| individualCount | 100.0 |
| recordedBy | 100.0 |
| recordNumber | 100.0 |
| occurrenceRemarks | 100.0 |

In [ ]:
# Full missingness table if needed
missingness = df.isna().mean().sort_values(ascending=False)
missingness.head(20)

## Taxonomic coverage

Top 10 scientific names by record count:

| count | records |
|---|---|
| Pomacentrus lepidogenys | 3,364 |
| Chlorurus sordidus | 3,359 |
| Acanthochromis polyacanthus | 3,275 |
| Scarus niger | 3,239 |
| Hemigymnus fasciatus | 2,765 |
| Plectropomus leopardus | 2,760 |
| Pomacentrus wardi | 2,746 |
| Chlorurus microrhinos | 2,725 |
| Pomacentrus moluccensis | 2,710 |
| Amblyglyphidodon curacao | 2,695 |

Top 10 families by record count:

| count | records |
|---|---|
| Pomacentridae | 55,693 |
| Scaridae | 33,410 |
| Chaetodontidae | 25,026 |
| Labridae | 16,654 |
| Acanthuridae | 14,827 |
| Siganidae | 7,677 |
| Lutjanidae | 5,459 |
| Serranidae | 5,076 |
| Lethrinidae | 3,143 |
| Zanclidae | 1,654 |

In [ ]:
# Top species and families
(df['scientificName'].value_counts().head(10),
 df['family'].value_counts().head(10))

## Temporal coverage

Counts for earliest years:

| count | records |
|---|---|
| 1992 | 3,038 |
| 1993 | 3,779 |
| 1994 | 4,963 |
| 1995 | 5,973 |
| 1996 | 6,002 |

Counts for latest years:

| count | records |
|---|---|
| 2014 | 5,570 |
| 2015 | 6,584 |
| 2016 | 7,664 |
| 2017 | 5,422 |
| 2018 | 5,213 |

In [ ]:
# Yearly trend
import matplotlib.pyplot as plt

year_counts = df['eventDate'].dt.year.value_counts().sort_index()
ax = year_counts.plot(kind='line', figsize=(8, 4), title='Records per year')
ax.set_xlabel('Year')
ax.set_ylabel('Records')

## Spatial coverage

The dataset spans the Great Barrier Reef region in Queensland, Australia. Use the plot below to visualize sampling locations.

In [ ]:
# Spatial scatter (lat/long)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df['decimalLongitude'], df['decimalLatitude'], s=4, alpha=0.2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Sampling locations')

## Blacktip reef shark check

No records found for **Carcharhinus melanopterus** (blacktip reef shark) or vernacular name 'blacktip reef shark'.

In [ ]:
# Blacktip reef shark check
blacktip_scientific = 'Carcharhinus melanopterus'
blacktip_mask = df['scientificName'].str.contains(blacktip_scientific, case=False, na=False)
blacktip_common_mask = df['vernacularName'].str.contains('blacktip reef shark', case=False, na=False)

blacktip_rows = df[blacktip_mask | blacktip_common_mask]
blacktip_rows[['eventDate', 'scientificName', 'vernacularName', 'decimalLatitude', 'decimalLongitude']].head()